# Evaluation: base vs fine-tuned

Scores the base Qwen2.5-1.5B-Instruct and the QLoRA-tuned version on the same 200 held-out examples. Two metrics:

- **exact match**: generated SQL == reference SQL, character for character
- **normalized match**: same but case-insensitive, whitespace collapsed, trailing `;` stripped. Fairer, since `SELECT Name` vs `select name` is the same query.

Needs a GPU (T4 is fine). 200 examples instead of the full 500 to keep generation time reasonable on the free tier.

In [ ]:
!pip install -q transformers==4.48.0 peft==0.14.0 bitsandbytes==0.45.0 accelerate==1.2.1 datasets==3.2.0

In [ ]:
import os

if not os.path.exists("text2sql-qlora"):
    !git clone https://github.com/Akshu24Tech/text2sql-qlora.git

from datasets import load_dataset

eval_ds = load_dataset("json", data_files="text2sql-qlora/data/eval.jsonl", split="train")
eval_ds = eval_ds.select(range(200))
eval_ds

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_ID = "Akshu24Tech/qwen25-1.5b-sql-lora"
SYSTEM = "You are a text-to-SQL assistant. Given a table schema and a question, reply with only the SQL query, nothing else."

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"  # needed for batched generation

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

In [ ]:
from tqdm.auto import tqdm

def build_prompt(row):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Schema:\n{row['context']}\n\nQuestion: {row['question']}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate_all(model, rows, batch_size=8):
    preds = []
    prompts = [build_prompt(r) for r in rows]
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch = prompts[i : i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True).to(model.device)
        out = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
        gen = out[:, inputs["input_ids"].shape[1]:]
        preds += [tokenizer.decode(g, skip_special_tokens=True).strip() for g in gen]
    return preds

rows = list(eval_ds)
base_preds = generate_all(model, rows)

In [ ]:
# same model object, adapter loaded on top
from peft import PeftModel

model = PeftModel.from_pretrained(model, ADAPTER_ID)
tuned_preds = generate_all(model, rows)

In [ ]:
import re

def normalize_sql(s):
    s = s.strip().rstrip(";").lower()
    s = re.sub(r"\s+", " ", s)
    return s

def score(preds, refs):
    exact = sum(p.strip() == r.strip() for p, r in zip(preds, refs))
    norm = sum(normalize_sql(p) == normalize_sql(r) for p, r in zip(preds, refs))
    n = len(refs)
    return {"exact_match": round(exact / n, 3), "normalized_match": round(norm / n, 3), "n": n}

refs = [r["answer"] for r in rows]
results = {
    "base": score(base_preds, refs),
    "fine_tuned": score(tuned_preds, refs),
}
results

In [ ]:
# look at a few cases where the tuned model fixed the base model
shown = 0
for r, b, t in zip(rows, base_preds, tuned_preds):
    ref = normalize_sql(r["answer"])
    if normalize_sql(t) == ref and normalize_sql(b) != ref:
        print("Q:", r["question"])
        print("ref :", r["answer"])
        print("base:", b)
        print("tuned:", t)
        print("-" * 80)
        shown += 1
    if shown == 5:
        break

In [ ]:
# and a few the tuned model still gets wrong, worth discussing in the report
shown = 0
for r, t in zip(rows, tuned_preds):
    if normalize_sql(t) != normalize_sql(r["answer"]):
        print("Q:", r["question"])
        print("ref :", r["answer"])
        print("tuned:", t)
        print("-" * 80)
        shown += 1
    if shown == 5:
        break

In [ ]:
import json

os.makedirs("text2sql-qlora/results", exist_ok=True)
with open("text2sql-qlora/results/metrics.json", "w") as f:
    json.dump(results, f, indent=2)

# dump raw predictions too, useful for the report
with open("text2sql-qlora/results/predictions.json", "w") as f:
    json.dump(
        [
            {"question": r["question"], "reference": r["answer"], "base": b, "tuned": t}
            for r, b, t in zip(rows, base_preds, tuned_preds)
        ],
        f,
        indent=2,
    )
print("saved")